<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/02_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - DPO preference tuning in Colab

This notebook uses Qwen's chat template and TRL's conversational preference format. This avoids manually concatenating `prompt + chosen/rejected`, which can cause tokenizer-prefix mismatch warnings.

This is a small teaching experiment, not a production RLHF pipeline.

In [ ]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"

In [ ]:
import torch
import transformers, datasets, peft, trl

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1. Preference data

TRL accepts conversational preference data directly. Each row contains a `prompt` conversation plus `chosen` and `rejected` assistant responses. The tokenizer's chat template is applied consistently by the trainer.

In [ ]:
from datasets import Dataset

preferences = [
    {'prompt': [{'role':'user','content':'Explain what DNA is.'}], 'chosen': [{'role':'assistant','content':'DNA is the molecule that stores hereditary genetic information in cells.'}], 'rejected': [{'role':'assistant','content':'DNA is a thing in biology related to genes.'}]},
    {'prompt': [{'role':'user','content':'Explain what RNA is.'}], 'chosen': [{'role':'assistant','content':'RNA is a nucleic acid involved in gene expression and several other cellular processes.'}], 'rejected': [{'role':'assistant','content':'RNA is basically DNA but different.'}]},
    {'prompt': [{'role':'user','content':'What is a gene?' }], 'chosen': [{'role':'assistant','content':'A gene is a DNA sequence that contributes to a functional product such as an RNA or protein.'}], 'rejected': [{'role':'assistant','content':'A gene is just a piece of DNA that causes a trait.'}]},
    {'prompt': [{'role':'user','content':'What is TP53?' }], 'chosen': [{'role':'assistant','content':'TP53 encodes p53, a tumor-suppressor protein that helps regulate cell-cycle responses to cellular stress and DNA damage.'}], 'rejected': [{'role':'assistant','content':'TP53 is a cancer gene that causes tumors.'}]},
    {'prompt': [{'role':'user','content':'What is BRCA1?' }], 'chosen': [{'role':'assistant','content':'BRCA1 encodes a protein involved in maintaining genome stability, including roles in DNA damage response and repair.'}], 'rejected': [{'role':'assistant','content':'BRCA1 is simply a gene that gives people breast cancer.'}]},
    {'prompt': [{'role':'user','content':'What is a protein?' }], 'chosen': [{'role':'assistant','content':'A protein is a polymer of amino acids that folds into structures enabling specific cellular functions.'}], 'rejected': [{'role':'assistant','content':'A protein is a type of molecule found in food and muscles.'}]},
    {'prompt': [{'role':'user','content':'What is a mutation?' }], 'chosen': [{'role':'assistant','content':'A mutation is a change in a DNA sequence; its biological effect depends on its type, location, and context.'}], 'rejected': [{'role':'assistant','content':'A mutation is always harmful.'}]},
    {'prompt': [{'role':'user','content':'What is gene expression?' }], 'chosen': [{'role':'assistant','content':'Gene expression is the process by which information encoded in a gene is used to produce a functional RNA or protein.'}], 'rejected': [{'role':'assistant','content':'Gene expression means a gene is turned on and makes DNA.'}]},
]
preferences = preferences * 12
dataset = Dataset.from_list(preferences)
dataset

In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']
print('Train rows:', len(train_dataset))
print('Eval rows:', len(eval_dataset))

## 2. Load the model and inspect the chat template

Qwen2.5-Instruct is a chat model, so we use `apply_chat_template` for generation. This is also the format used by TRL for the preference dataset.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

print('Has chat template:', tokenizer.chat_template is not None)
print(tokenizer.apply_chat_template(train_dataset[0]['prompt'], tokenize=False, add_generation_prompt=True))

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model = model.cuda()

## 3. Baseline generation

This is the model before DPO.

In [ ]:
def generate(model, question, max_new_tokens=80):
    messages = [{'role':'user','content':question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_QUESTION = 'Why is BRCA1 biologically important?'
print('BEFORE DPO')
print(generate(model, TEST_QUESTION))

## 4. DPO + LoRA

Only the LoRA adapter is trained. Because the dataset is conversational and uses Qwen's chat template, TRL can consistently identify the prompt portion and the chosen/rejected completion.

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

dpo_args = DPOConfig(
    output_dir='./outputs/dpo-biobot',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-5,
    beta=0.1,
    max_length=256,
    max_prompt_length=128,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

In [ ]:
model.eval()
print('AFTER DPO')
print(generate(model, TEST_QUESTION))

## 5. Try several prompts

Compare the style before and after DPO. The tiny dataset teaches a preference pattern, not new authoritative biomedical knowledge.

In [ ]:
for q in [
    'What does TP53 do?',
    'What is a mutation?',
    'What is gene expression?',
    'Why is BRCA1 biologically important?',
]:
    print('\nQUESTION:', q)
    print(generate(model, q))

In [ ]:
ADAPTER_DIR = './outputs/dpo-biobot-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:', ADAPTER_DIR)

## What to learn from this experiment

**SFT / LoRA:** the model sees target answers and learns to imitate them.

**DPO:** the model sees a preferred answer and a less-preferred answer and learns their relative preference.

Next, compare **Base → SFT/LoRA → DPO** using the same evaluation prompts. After that, we can explore reward models and PPO-style RLHF.